# Fundamentals 08 - System API

Objetivo: mostrar `AgenticSystem` como system y fabrica de composicion. Un system no es un agente: registra tools, skills, agents, runtime y contratos para que la ejecucion sea auditable y repetible.

Modelo mental:

```text
System = registry + runtime + skills + agents + inspect
```


<!-- system-language -->
## Lenguaje canonico

En estos tutorials usamos `system` como nombre publico para `AgenticSystem`: el contenedor que registra tools, skills, agents, runtime y contratos. `Toolkit` existe como agrupacion avanzada de tools dentro del system, pero la primitiva que se ensena primero sigue siendo `Tool`.


In [ ]:
import agentic_systems as toolkit

PRETTY = False

scheduler = toolkit.scheduler(timeout_s=30, max_retries=0, max_tool_calls=8, max_turns=8)
runtime = toolkit.runtime(provider="python-runtime", model="python-runtime", region="local", scheduler=scheduler)

system = toolkit.AgenticSystem(runtime=runtime)

## Parametros de `RunPolicy`

`RunPolicy` declara como debe comportarse una ejecucion antes de llamar al agente o al runtime. No es metadata decorativa: limita loops, define reparacion, controla trazas y hace que el resultado sea evaluable.

| Parametro | Que controla | Uso recomendado |
|---|---|---|
| `max_turns` | Numero maximo de turnos internos del agente. | Mantenerlo bajo en notebooks para evitar loops largos. |
| `max_tool_calls` | Numero maximo de llamadas a tools. | Declararlo cuando el ejercicio espera tools concretas. |
| `max_tokens` | Limite de tokens del modelo cuando el provider lo soporta. | util en providers LM; puede quedar `None` en `python-runtime`. |
| `temperature` | Aleatoriedad del modelo. | `0.0` para tutoriales reproducibles; `None` delega al provider. |
| `tool_choice` | Estrategia de seleccion de tools, por ejemplo `auto`. | `auto` cuando el agente decide; explicito cuando quieres forzar una tool. |
| `repair` | Permite reparacion automatica de salidas o tool calls invalidas. | `True` para UX robusta; `False` si quieres ver fallos crudos. |
| `max_repairs` | Maximo de intentos de reparacion. | `1` o `2` en tutoriales para mostrar control sin ocultar errores. |
| `finalize` | Que hacer al agotar turnos, por ejemplo `on_max_turns`. | Mantenerlo explicito en agentes LM evaluables. |
| `trace` | Nivel de trazabilidad (`compact`, `debug`, etc.). | `compact` para notebooks; `debug` solo para diagnostico. |
| `strict` | Si el contrato debe aplicarse de forma estricta. | `True` para ensenar API y evitar ambiguedad. |


## Escenario didactico compartido

Todos los notebooks de `tutorials/` usan este mismo problema para comparar la API sin cambiar de caso:

```text
Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final
```


In [ ]:
USER_PROMPT = """Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final""".strip()

REQUESTED_OUTPUTS = ["procedimiento", "resultado_final"]

toolkit.show({
    "prompt_usuario": USER_PROMPT,
    "salidas_solicitadas": REQUESTED_OUTPUTS,
}, title="Escenario didactico")

## 1) Registrar tools en el system

`system.tool(...)` mete la tool en el registry del system. Esto sirve cuando varias skills o agentes deben compartir la misma capacidad sin duplicar definiciones.

In [ ]:
@system.tool
def sumar(a: int, b: int) -> dict:
    return {"operation": "sumar", "result": a + b, "explanation": f"{a} + {b} = {a + b}"}


@system.tool
def restar(a: int, b: int) -> dict:
    return {"operation": "restar", "result": a - b, "explanation": f"{a} - {b} = {a - b}"}


@system.tool
def multiplicar(a: int, b: int) -> dict:
    return {"operation": "multiplicar", "result": a * b, "explanation": f"{a} * {b} = {a * b}"}


@system.tool
def dividir(a: float, b: float) -> dict:
    if b == 0:
        raise ValueError("No se puede dividir entre cero.")
    value = a / b
    return {"operation": "dividir", "result": value, "explanation": f"{a} / {b} = {value}"}

toolkit.show({
    "tool_names": list(system.public_tool_names),
    "runtime_tool_names": list(system.tool_names),
}, title="System registry")

## 2) Empaquetar esas tools como skill runtime

La skill da nombre, prompts, contratos y policy a un paquete de tools. El system la puede registrar y expandir en agentes sin perder el contrato.

In [ ]:
calculator_contract = toolkit.AgentContract(
    must_call=["sumar", "restar", "multiplicar", "dividir"],
    completion="when_required_tools_satisfied",
)
calculator_policy = toolkit.RunPolicy(max_tool_calls=4, max_turns=4)

math_skill = toolkit.Skill(
    name="system_math",
    description="Skill aritmetica compartida por el system.",
    tools=[system.public_tools["sumar"], system.public_tools["restar"], system.public_tools["multiplicar"], system.public_tools["dividir"]],
    prompts={"instructions": "Usa tools aritmeticas y conserva evidencia estructurada."},
    contracts={"default": calculator_contract.model_dump(mode="json")},
    policy=calculator_policy.model_dump(mode="json"),
)

system.skill(math_skill)

toolkit.show({
    "skill_names": list(system.skill_names),
    "runtime_skills": [skill.info() for skill in system.runtime_skills],
}, title="System skills")

## 3) Crear un agente desde el system

`system.agent(...)` resuelve tools y skills desde el system. El agente queda ligado al runtime del system y puede ejecutarse con `python-runtime` para pruebas deterministas.

In [ ]:
single_step_contract = toolkit.AgentContract(must_call=["sumar"], completion="when_required_tools_satisfied")

calculator_agent = system.agent(
    name="system_calculator_agent",
    instructions=math_skill.instructions,
    skills=[math_skill],
    engine="python-runtime",
    contract=single_step_contract,
    policy=toolkit.RunPolicy(max_tool_calls=1, max_turns=1),
    runtime=runtime,
)

result = calculator_agent.run({"tool": "sumar", "input": {"a": 10, "b": 20}}, mode="eval")

toolkit.human_result(
    result,
    title="Human result - system.agent(...)",
    expected_tools=toolkit.expect.exactly("sumar"),
    pretty=PRETTY,
)

## 4) Pipeline determinista dentro del system

Un pipeline no necesita un loop reactivo. Puede ejecutar tools registradas en orden fijo, conservar evidencia y producir un resultado auditable con `compose_result`.

In [ ]:
operation_plan = [
    ("sumar", {"a": 10, "b": 20}),
    ("restar", {"a": 30, "b": 9}),
    ("multiplicar", {"a": 21, "b": 4}),
    ("dividir", {"a": 84, "b": 2}),
]

operation_trace = []
tool_results = []
for tool_name, tool_input in operation_plan:
    tool_result = system.public_tools[tool_name].run(tool_input)
    operation_trace.append(tool_result.data)
    tool_results.append(tool_result)

pipeline_final = {
    "procedimiento": [item["explanation"] for item in operation_trace],
    "resultado_final": operation_trace[-1]["result"],
}

pipeline_result = toolkit.compose_result(
    text="El pipeline determinista resolvio el escenario didactico.",
    data=pipeline_final,
    results=tool_results,
    mode="pipeline",
    input=USER_PROMPT,
    meta={"system": "fundamentals", "operation_trace": operation_trace},
)

toolkit.human_result(
    pipeline_result,
    title="Human result - deterministic system pipeline",
    pretty=PRETTY,
)

## 5) Inspeccionar el System sin ejecutar

system.inspect() produce InspectReport antes de ejecutar modelos o Tools. La
proyeccion estructurada expone entidades, relaciones, contratos, Providers,
Frameworks, capacidades, conflictos, limites y riesgos de degradacion.

In [ ]:
inspection = system.inspect()
inspection.raise_if_errors()

toolkit.show(inspection.to_dict(), title="System inspect - structured")
print(inspection.human_text())

assert inspection["side_effects"] == {
    "models_executed": 0,
    "tools_executed": 0,
}

## Lo importante

- AgenticSystem registra Tools, Skills, Agents y Runtime.
- system.agent(...) liga el Agent al registry y Runtime del System.
- Un pipeline determinista es composicion explicita de Tools.
- system.inspect() es una observacion estatica, serializable y no ejecutable.
- InspectReport.human_text() y to_dict() proyectan el mismo reporte.

## Coverage API de este notebook

Esta tabla deja explicito que parte de Agentic Systems queda materializada aqui.

In [ ]:
api_coverage = [
    {"api": "toolkit.AgenticSystem", "description": "Crea el system y fabrica de tools, skills, agents y runtime."},
    {"api": "system.tool", "description": "Registra tools reutilizables en el registry del system."},
    {"api": "system.skill", "description": "Registra una Skill runtime y expande sus tools."},
    {"api": "system.agent", "description": "Crea agentes ligados al registry y runtime del system."},
    {"api": "deterministic pipeline", "description": "Ejecuta tools en orden fijo sin introducir un parametro loop nuevo."},
    {"api": "system.inspect", "description": "Audita estaticamente el System sin ejecutar modelos ni Tools."},
    {"api": "InspectReport.to_dict", "description": "Produce salida estructurada serializable."},
    {"api": "InspectReport.human_text", "description": "Produce salida humana estable."},
]

toolkit.show({"notebook": "08_system_api.ipynb", "api_coverage": api_coverage})

## Simbolos API explicados

Este notebook se alinea con `docs/API.md` y ensena estos simbolos publicos:

- `AgenticSystem`: System nativo para tools, skills, agentes y pipelines.
- `toolkit.compose_result`: Resultado compuesto para pipelines deterministas.
- `core / providers / integrations`: Namespaces publicos que delimitan capas.
- `Skill`: Skill registrada dentro del system.
- `human_result`: Render del resultado del system.


- InspectReport: reporte publico y compatible con dict.
- InspectReport.to_dict / human_text: proyecciones estructurada y humana.
